# 4. Dynamics: MSD & Diffusion

Time-dependent properties describe how a system *moves*. This chapter computes the
**mean squared displacement (MSD)** of water and estimates a **diffusion
coefficient** from an MSD curve.

```{admonition} A note on cost
:class: note
MSD is computed over all selected molecules and time origins, so it is heavier
than the per-frame analyses. On the small bundled BPTI system it still runs in
seconds.
```


In [ ]:
import numpy as np
from genepie import genesis_exe, SMolecule
from genepie.tests.conftest import BPTI_PDB, BPTI_PSF, BPTI_DCD, MSD_DATA

mol = SMolecule.from_file(pdb=BPTI_PDB, psf=BPTI_PSF)
trajs, _ = genesis_exe.crd_convert(
    mol, trj_files=[str(BPTI_DCD)], trj_format="DCD",
    trj_type="COOR+BOX", selection="all",
)

## Mean squared displacement of water

We track the TIP3 water molecules (`mode=["ALL"]` = 3-D MSD).

In [ ]:
msd = genesis_exe.msd_analysis(
    mol, trajs[0],
    selection_group=["rnam:TIP3"],
    selection=[1],
    mode=["ALL"],
    oversample=True,
    delta=9,
).msd
print("MSD shape (lag, group):", msd.shape)
print("MSD(t): ", np.round(msd[:, 0], 3))

## Diffusion coefficient from an MSD curve

`diffusion_analysis` fits the linear (Einstein) regime of an MSD curve. Here we
use a longer pre-computed MSD dataset bundled with the tests. The Einstein
relation gives $D$ from the slope; the second half of the curve is used for the fit.


In [ ]:
msd_data = np.loadtxt(MSD_DATA, dtype=np.float64)   # column 0 = time, rest = MSD
res = genesis_exe.diffusion_analysis(
    msd_data, time_step=2.0, start_step=int(msd_data.shape[0] * 0.2),
)
print("diffusion coefficient(s):", res.diffusion_coefficients)
print("fit output shape:", res.out_data.shape)

## Plot MSD and its linear fit

In [ ]:
import plotly.io as pio
import plotly.graph_objects as go
pio.renderers.default = "notebook"

t = res.out_data[:, 0]
fig = go.Figure()
fig.add_trace(go.Scatter(x=t, y=res.out_data[:, 1], mode="markers", name="MSD"))
fig.add_trace(go.Scatter(x=t, y=res.out_data[:, 2], mode="lines", name="linear fit"))
fig.update_layout(title="MSD and Einstein-relation fit", xaxis_title="time",
                  yaxis_title="MSD", template="plotly_white", height=380)
fig